# Etapa 5 — Redes Bayesianas

Nesta etapa, vamos modelar as dependências entre as variáveis do domínio do Meio Ambiente usando uma Rede Bayesiana. O objetivo é criar um Grafo Acíclico Direcionado (DAG), estimar as Tabelas de Probabilidade Condicional (CPTs) a partir dos dados e realizar inferências.

**Bibliotecas necessárias:**
Usaremos o `pgmpy` para a modelagem e inferência, além do `pandas` para manipular os dados.


In [43]:
import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination


print("Carregando os dados...")
df = pd.read_csv("../dados/tabela_p4.csv", encoding="utf-8-sig")

# Selecionar apenas as 5 variáveis que vamos usar na nossa Rede Bayesiana
variaveis_rede = ['CHUVA_FAIXA', 'TEMP_FAIXA', 'BIOMA', 'TIPO_ACAO', 'Multa_Alta']
df = df[variaveis_rede].dropna()

# Convertendo tudo para string (categorias) para facilitar o pgmpy
df = df.astype(str)

print(f"Dados carregados! Total de linhas: {len(df)}")
df.head()

Carregando os dados...
Dados carregados! Total de linhas: 2618


,CHUVA_FAIXA,TEMP_FAIXA,BIOMA,TIPO_ACAO,Multa_Alta
0,alta,alta,Amazonia,Rotina,0
1,media,baixa,Mata Atlantica,Operação,0
2,media,alta,Caatinga,Rotina,0
3,media,alta,Caatinga,Rotina,0
4,media,alta,Caatinga,Rotina,0


## Passo 1 — Desenhar o DAG (Grafo Acíclico Direcionado)
Precisamos definir de 4 a 6 variáveis e justificar as arestas (causa $\rightarrow$ efeito).
Escolhemos 5 variáveis:
1. **CHUVA_FAIXA**: Condição climática (chuva).
2. **TEMP_FAIXA**: Condição climática (temperatura).
3. **BIOMA**: Região da ocorrência.
4. **TIPO_ACAO**: Tipo de infração/ação do IBAMA (ex: Rotina, Operação).
5. **Multa_Alta**: Nossa variável alvo (0 ou 1).

**Justificativa das Arestas:**
* Clima (`CHUVA_FAIXA` e `TEMP_FAIXA`) e o `BIOMA` influenciam diretamente as condições do terreno e as atividades na região, afetando o `TIPO_ACAO`.
* O `TIPO_ACAO` (gravidade ou foco da fiscalização) influencia diretamente a probabilidade de aplicação de uma `Multa_Alta`.

In [44]:
# Criando a estrutura do modelo
modelo = DiscreteBayesianNetwork([
    ('CHUVA_FAIXA', 'TIPO_ACAO'),
    ('TEMP_FAIXA', 'TIPO_ACAO'),
    ('BIOMA', 'TIPO_ACAO'),
    ('TIPO_ACAO', 'Multa_Alta')
])

print("Nós da rede:", modelo.nodes())
print("Arestas criadas:", modelo.edges())

Nós da rede: ['CHUVA_FAIXA', 'TIPO_ACAO', 'TEMP_FAIXA', 'BIOMA', 'Multa_Alta']
Arestas criadas: [('CHUVA_FAIXA', 'TIPO_ACAO'), ('TIPO_ACAO', 'Multa_Alta'), ('TEMP_FAIXA', 'TIPO_ACAO'), ('BIOMA', 'TIPO_ACAO')]


## Passo 2 — Estimar as CPTs a partir dos dados

Com o grafo desenhado, a rede precisa aprender as probabilidades. Como temos a tabela de dados completa, usamos a Estimativa de Máxima Verossimilhança (Maximum Likelihood Estimation - MLE). O `pgmpy` conta as frequências no DataFrame e constrói as tabelas.

In [45]:
# Aprendendo as probabilidades a partir dos dados reais (usa MLE por defeito)
modelo.fit(df)

# Validar se o modelo foi construído corretamente (soma das probabilidades = 1)
print("Modelo é válido?", modelo.check_model())

# Vamos visualizar a CPT de uma das variáveis (ex: Multa_Alta dependendo do TIPO_ACAO)
cpt_multa = modelo.get_cpds('Multa_Alta')
print("\nTabela de Probabilidade Condicional para Multa_Alta:")
print(cpt_multa)

Modelo é válido? True

Tabela de Probabilidade Condicional para Multa_Alta:
+---------------+-----+---------------------+
| TIPO_ACAO     | ... | TIPO_ACAO(Rotina)   |
+---------------+-----+---------------------+
| Multa_Alta(0) | ... | 0.47761194029850745 |
+---------------+-----+---------------------+
| Multa_Alta(1) | ... | 0.5223880597014925  |
+---------------+-----+---------------------+


## Passo 3 — Inferência (Respondendo a consultas)

Vamos usar a Eliminação de Variáveis (`VariableElimination` do `pgmpy`) para responder a perguntas sobre o nosso modelo.

**Consulta:** Qual a probabilidade de a multa ser alta (`Multa_Alta`) sabendo que a ação do IBAMA foi uma "Operação"?

In [46]:
from pgmpy.inference import VariableElimination

# Criar o motor de inferência
inferencia = VariableElimination(modelo)

# Calcular a probabilidade
resultado = inferencia.query(variables=['Multa_Alta'], evidence={'TIPO_ACAO': 'Operação'})
print("Probabilidade de Multa Alta dado que a ação é uma 'Operação':")
print(resultado)

Probabilidade de Multa Alta dado que a ação é uma 'Operação':
+---------------+-------------------+
| Multa_Alta    |   phi(Multa_Alta) |
+===============+===================+
| Multa_Alta(0) |            0.5122 |
+---------------+-------------------+
| Multa_Alta(1) |            0.4878 |
+---------------+-------------------+


## Passo 4 — Independência Condicional (d-separação)

A d-separação permite verificar se duas variáveis são independentes dado o conhecimento de uma terceira. 

No nosso modelo, o `TIPO_ACAO` está no meio do caminho entre o clima (`CHUVA_FAIXA`) e a `Multa_Alta`. Vamos provar que se soubermos o `TIPO_ACAO`, a chuva deixa de influenciar diretamente a probabilidade da multa (o caminho fica bloqueado).

In [47]:
# Teste de d-separação usando a verificação de independência do pgmpy
# Vamos testar se 'CHUVA_FAIXA' e 'Multa_Alta' são d-separados dado 'TIPO_ACAO'

# 1. Sem evidência (deve estar dependente / caminho aberto)
independente_sem_evidencia = modelo.is_dconnected('CHUVA_FAIXA', 'Multa_Alta')
print(f"Caminho ativo entre Chuva e Multa (sem evidência): {independente_sem_evidencia}")

# 2. Com evidência no nó intermédio 'TIPO_ACAO' (deve bloquear o caminho)
independente_com_evidencia = modelo.is_dconnected('CHUVA_FAIXA', 'Multa_Alta', observed=['TIPO_ACAO'])
print(f"Caminho ativo entre Chuva e Multa (sabendo o TIPO_ACAO): {independente_com_evidencia}")

Caminho ativo entre Chuva e Multa (sem evidência): True
Caminho ativo entre Chuva e Multa (sabendo o TIPO_ACAO): False


## Passo 5 — Economia de Parâmetros

Uma das grandes vantagens das Redes Bayesianas é a redução do número de parâmetros (números) necessários em comparação com uma tabela de distribuição conjunta completa.

Vamos calcular quantos parâmetros a nossa rede economizou.

In [48]:
import numpy as np

# Calcular parâmetros da distribuição conjunta completa
categorias = {var: modelo.get_cardinality(var) for var in modelo.nodes()}
total_conjunta = np.prod(list(categorias.values())) - 1

# Calcular parâmetros da Rede Bayesiana
total_rede = sum([len(cpt.values.flatten()) - (len(cpt.values.flatten()) // cpt.variable_card) for cpt in modelo.get_cpds()])

print(f"Parâmetros necessários na Conjunta Completa: {total_conjunta}")
print(f"Parâmetros necessários na Rede Bayesiana: {total_rede}")
print(f"Economia total: a nossa rede poupa {total_conjunta - total_rede} parâmetros!")

Parâmetros necessários na Conjunta Completa: 671
Parâmetros necessários na Rede Bayesiana: 239
Economia total: a nossa rede poupa 432 parâmetros!
